# Multimodal SSL + DA + CL

This notebook reproduces the final multimodal self-supervised learning strategy described in the thesis. The experiment has two stages:

- **SSL pretraining:** paired clinical--dermoscopic images from the same lesion are used as positive pairs; other lesions in the batch act as negatives.
- **Supervised fine-tuning:** the pretrained encoder is fine-tuned on the dermoscopic task using the final supervised interventions: online data augmentation, class-aware repeat factor, class-only weighted sampling, and class--skin-tone weighted loss.

The notebook intentionally delegates the implementation to the repository modules and YAML files, so the executable configuration remains identical to the code used by the scripts.


## Configuration Loading

The final thesis setting evaluates the SSL-based strategy on the selected **ViT-B/32 dermoscopic** configuration for both task formulations. The SSL contrastive stage uses the symmetric multimodal NT-Xent objective with temperature \(\tau = 0.10\).


In [ ]:
from pathlib import Path
import sys

# Make the notebook robust whether it is launched from the repository root or from notebooks/.
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from milk10k.training.ssl_multimodal import run as run_ssl_aug_plus_loss
from milk10k.utils.config import load_config

CONFIG_PATHS = {
    "11f": PROJECT_ROOT / "configs" / "ssl.yaml",
    "2f": PROJECT_ROOT / "configs" / "ssl_2f.yaml",
}

configs = {task: load_config(path) for task, path in CONFIG_PATHS.items()}
configs


## Consistency Checks

These assertions encode the final experimental decisions reported in the thesis. If one of them fails, the notebook configuration is no longer aligned with the written methodology.


In [ ]:
expected = {
    "11f": {"batch_size": 128, "beta": 0.999},
    "2f": {"batch_size": 64, "beta": 0.99},
}

for task, cfg in configs.items():
    assert cfg.task == task
    assert cfg.backbone == "vit_b_32"
    assert cfg.data.modality == "dermoscopic"
    assert cfg.pretrained is True
    assert cfg.image_size == 224
    assert cfg.lr == 3e-5
    assert cfg.ssl_lr == 3e-5
    assert cfg.ssl_batch_size == 64
    assert cfg.temperature == 0.10
    assert cfg.weighted_sampler is True
    assert cfg.repeat_exponent == 0.5
    assert cfg.repeat_cap == 6
    assert cfg.alpha == 0.4
    assert cfg.batch_size == expected[task]["batch_size"]
    assert cfg.beta == expected[task]["beta"]

summary = [
    {
        "task": task,
        "modality": cfg.data.modality,
        "backbone": cfg.backbone,
        "ssl_temperature": cfg.temperature,
        "ssl_batch_size": cfg.ssl_batch_size,
        "finetune_batch_size": cfg.batch_size,
        "beta": cfg.beta,
        "weighted_sampler": cfg.weighted_sampler,
        "repeat_factor": f"sqrt imbalance, cap={cfg.repeat_cap}",
        "output_dir": cfg.output_dir,
    }
    for task, cfg in configs.items()
]
summary


## Execute Final SSL-Based Experiments

Execution is disabled by default to avoid launching long training runs accidentally. Set the corresponding flag to `True` to run one task. Each run performs multimodal SSL pretraining first and then supervised dermoscopic fine-tuning with the final DA+CL configuration.


In [ ]:
RUN_11F = False
RUN_2F = False

if RUN_11F:
    run_ssl_aug_plus_loss(configs["11f"])

if RUN_2F:
    run_ssl_aug_plus_loss(configs["2f"])
